# 02 Features & Models

Shows temporal split, leakage test results, and model evaluation from pre-run artifacts.

In [1]:
import pandas as pd, json, warnings; warnings.filterwarnings('ignore')
df = pd.read_parquet('../data/processed/features.parquet')
for s in ['train', 'val', 'test']:
    sub = df[df['split'] == s]
    print(f"{s.upper():5}: {sub['date'].min()} to {sub['date'].max()} | {len(sub):5} rows | SO rate: {sub['stockout_flag'].mean()*100:.1f}%")


TRAIN: 2026-05-15 00:00:00 to 2026-08-05 00:00:00 | 10624 rows | SO rate: 41.7%
VAL  : 2026-08-13 00:00:00 to 2026-08-30 00:00:00 |  2304 rows | SO rate: 49.1%
TEST : 2026-09-07 00:00:00 to 2026-09-23 00:00:00 |  2201 rows | SO rate: 40.8%


## Leakage Tests (A–E)

In [2]:
import sys; sys.path.insert(0, '..')
from tests.verify_phase7a import run_checks
run_checks()


Leakage Tests: A=PASS | B=PASS | C=PASS | D=PASS | E=PASS
TRAIN: 2026-05-15 to 2026-08-05 (10624 rows, 83d) | SO Rate: 41.68%
VAL: 2026-08-13 to 2026-08-30 (2304 rows, 18d) | SO Rate: 49.13%
TEST: 2026-09-07 to 2026-09-23 (2201 rows, 17d) | SO Rate: 40.80%
Demand Test MAE: 11.71 vs Baseline: 14.08 (Delta: +0.02)
Stockout Recall: 0.7595 vs Baseline: 0.5902 (Delta: +0.0000)
Stockout ROC-AUC: 0.8358 vs Baseline: 0.6017 (Delta: +0.0009)


## Model Comparison Table

In [3]:
comp = pd.read_csv('../reports/model_comparison.csv')
demand = comp[comp['task']=='Demand Regression'][['model','MAE','RMSE','R2']]
print('Demand Models (Test):\n', demand.to_string(index=False))

Demand Models (Test):
                       model      MAE      RMSE            R2
Baseline (Rolling Mean 7x7)    14.08     18.95  9.564000e-01
          Linear Regression 37263.23 616207.18 -4.605250e+07
              Decision Tree    13.52     18.53  9.584000e-01
              Random Forest    12.02     16.59  9.666000e-01
                    XGBoost    11.71     16.10  9.686000e-01
                        KNN    16.60     23.12  9.352000e-01


In [4]:
stockout = comp[comp['task']=='Stockout Classification'][['model','Recall','ROC_AUC','F1']]
print('Stockout Models (Test):\n', stockout.to_string(index=False))

Stockout Models (Test):
                            model  Recall  ROC_AUC     F1
Baseline (closing < reorder_lvl)  0.5902   0.6017 0.5487
                   Random Forest  0.7450   0.8306 0.7079
                         XGBoost  0.7595   0.8358 0.7078
                   Decision Tree  0.7116   0.8089 0.6670
                             KNN  0.5145   0.7585 0.5859
                     Naive Bayes  0.6414   0.7170 0.6247


## Risk Tier Calibration (Trust Slide)

In [5]:
with open('../models/metrics.json') as f: m = json.load(f)
for tier in m['stockout_model']['tier_calibration']:
    print(f"  {tier['risk_tier']:6}: {tier['pct_share']:.1f}% of test set → actual stockout rate {tier['actual_stockout_rate']*100:.1f}%")


  LOW   : 46.3% of test set → actual stockout rate 13.6%
  MEDIUM: 25.4% of test set → actual stockout rate 49.8%
  HIGH  : 28.3% of test set → actual stockout rate 77.2%
